# Teks — Klasifikasi dengan Embedding dari Ollama (Qwen3-Embedding)
**Kapan dipakai:** sama seperti `klasifikasi_embedding.ipynb`, tetapi model embedding
dijalankan oleh **Ollama** (bukan `sentence_transformers`).

**Syarat:** aplikasi Ollama menyala dan model sudah diunduh,
mis. `ollama pull qwen3-embedding:0.6b`.

**Langkah:** teks → vektor lewat API Ollama (sekali, disimpan ke `.npy`)
→ classifier sklearn → F1 macro.

**Data demo:** SmSA (sentimen Bahasa Indonesia).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from tqdm.auto import tqdm

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"   # UBAH saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"
OUTPUT_FOLDER = "../../../outputs/text/embedding_ollama"

OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "qwen3-embedding:0.6b"     # UBAH: paling ringan (639 MB)
# EMBEDDING_MODEL = "qwen3-embedding:4b"     # lebih kuat, 2,5 GB
# EMBEDDING_MODEL = "qwen3-embedding:8b"     # terkuat, 4,7 GB

TEXT_PREFIX = ""
# Qwen3-Embedding bisa diberi instruksi tugas (sering menaikkan skor beberapa poin):
# TEXT_PREFIX = "Instruct: Classify the sentiment of the review\nQuery: "

TEXT_COL = "text"
TARGET = "label"
BATCH_SIZE = 64      # turunkan bila Ollama kehabisan memori
SEED = 42
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

## Cek Ollama
Kalau sel ini error "Connection refused", buka aplikasi Ollama dulu. Nama model yang dipakai harus
ada di daftar ini.

In [ ]:
installed = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json()["models"]
for item in installed:
    print(item["name"])

## Baca data

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
train[TEXT_COL] = train[TEXT_COL].fillna("")
test[TEXT_COL] = test[TEXT_COL].fillna("")
print("train:", train.shape, "| test:", test.shape)
print(train[TARGET].value_counts())
train.head()

## Embedding lewat Ollama
Teks dikirim per batch ke `/api/embed`. Hasilnya disimpan ke `.npy` di `OUTPUT_FOLDER`,
jadi Run All berikutnya tidak menghitung ulang. Hapus file `.npy` itu bila data, model,
atau `TEXT_PREFIX` diganti.

In [ ]:
def embed(texts):
    vectors = []
    for start in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[start:start + BATCH_SIZE]
        body = {"model": EMBEDDING_MODEL, "input": batch, "truncate": True}
        response = requests.post(f"{OLLAMA_URL}/api/embed", json=body, timeout=600)
        response.raise_for_status()
        vectors.extend(response.json()["embeddings"])
    return np.array(vectors, dtype=np.float32)


def embed_cached(texts, name):
    path = f"{OUTPUT_FOLDER}/{name}_{EMBEDDING_MODEL.replace(':', '_')}.npy"
    if os.path.exists(path):
        return np.load(path)
    vectors = embed(texts)
    np.save(path, vectors)
    return vectors


train_vectors = embed_cached(list(TEXT_PREFIX + train[TEXT_COL]), "train")
test_vectors = embed_cached(list(TEXT_PREFIX + test[TEXT_COL]), "test")
print("train:", train_vectors.shape, "| test:", test_vectors.shape)

## Split train / validasi

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_vectors, train[TARGET], test_size=0.2, stratify=train[TARGET], random_state=SEED
)

## Model
Pilih SATU model: hapus `#` di baris yang ingin dipakai, beri `#` pada baris lain.

In [ ]:
model = LogisticRegression(C=1.0, max_iter=2000)                 # cepat, mudah dijelaskan
# model = LinearSVC(C=0.5)
# model = KNeighborsClassifier(n_neighbors=15, metric="cosine")
# model = MLPClassifier(hidden_layer_sizes=(256,), max_iter=300, random_state=SEED)
# model = SVC(C=10, random_state=SEED)
# model = RidgeClassifier(alpha=1.0)
# model = LGBMClassifier(n_estimators=300, verbose=-1, random_state=SEED)

pipe = make_pipeline(StandardScaler(), model)
pipe.fit(X_train, y_train)

## Evaluasi di validasi

In [ ]:
val_predictions = pipe.predict(X_val)
print("F1 macro:", round(f1_score(y_val, val_predictions, average="macro"), 4))
print(classification_report(y_val, val_predictions))
ConfusionMatrixDisplay.from_predictions(y_val, val_predictions, xticks_rotation=45)
plt.show()

## Latih ulang di semua data + submission

In [ ]:
pipe.fit(train_vectors, train[TARGET])
predictions = pipe.predict(test_vectors)

submission = pd.DataFrame()
submission["id"] = test["id"]
submission["label"] = predictions
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
submission.head()